In [1]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 16.6 MB/s eta 0:00:00


In [2]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('1_LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('1_LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [3]:
import os
from pyngrok import ngrok

In [4]:
ngrok.kill()

In [5]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://sporoid-nonnegligibly-casie.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://sporoid-nonnegligibly-casie.ngrok-free.dev


True

In [6]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        response_modalities=["TEXT"],
    )
)

In [10]:
def stateful_query(payload):
    response = chat.send_message(message=payload)  #把先前的信息一起送進去問
    return response.text

In [8]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Ming Hsin University of Science and Technology，簡稱明新科大）是一所位於台灣新竹縣的私立科技大學。

以下是其簡要介紹：

1.  **創立與沿革：** 創立於1966年，前身為「明新工業專科學校」，經過多年的發展與擴充，於2001年改制為「明新科技大學」。其歷史悠久，是台灣較早期的私立技職院校之一。

2.  **地理位置與特色：** 學校坐落於新竹縣新豐鄉，鄰近新竹科學園區、湖口工業區等科技與產業聚落，享有地利之便。這使得明新科大能與周邊產業建立緊密的產學合作關係，提供學生豐富的實習與就業機會，也讓其課程設計更貼近產業需求。

3.  **教育理念：** 明新科大以「誠、樸、精、勤」為校訓，強調理論與實務並重，致力於培養具備專業技能、創新思維及良好職業倫理的應用型人才，以符合國家經濟發展及社會變遷所需。

4.  **學術架構：** 目前設有以下四大學院：
    *   **工程學院：** 涵蓋電子、電機、機械、土木、資訊工程等領域。
    *   **管理學院：** 包含企業管理、資訊管理、財務金融、國際企業等。
    *   **服務事業學院：** 設有旅館管理、餐飲管理、幼兒保育、運動管理等系所。
    *   **人文社會學院：** 提供應用外語、多媒體與遊戲設計等課程。

5.  **發展重點：** 明新科大在教學上強調實務操作與證照取得，並積極推動國際交流與產學合作，提升學生的國際視野與就業競爭力。近年來，也積極投入智慧科技、綠能環保等領域的研發與人才培育。

總體而言，明新科技大學是一所深耕在地、與產業緊密結合，以培育實用型科技與服務人才為核心使命的綜合型科技大學。


In [11]:
result2 = stateful_query("校長是誰？")
print(result2)

截至我目前的知識更新（通常是到2023年底，若無特別註明），明新科技大學的現任校長是 **劉國偉** 教授。

您可以隨時查閱明新科技大學的官方網站以獲取最即時的校長資訊。


In [12]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [22/May/2026 02:52:08] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U3169a962e22ea2afc45f6eb98941b9c1","events":[]}
BODY:  {"destination":"U3169a962e22ea2afc45f6eb98941b9c1","events":[{"type":"message","message":{"type":"text","id":"615031053958512796","quoteToken":"CmPugqYc7p5vyqLS0nGmIC8b8And08kAPZK0fdrknoZvnTCXt6P-W1aJ3vN_KMYmWcANPwZHQycBpKu8EkolSYnqjQ9XfNilcwsp81JfBrVAODJElkghj1BBoFqG_yQ0SJ3HjpbsH3G33zzdV5iy5A","markAsReadToken":"l2owl56Th468tBr2fbiD3dVh7vM1rcbnceVm7TX5E3dd5de1D0vt8PCi6gbXiEEq8ljqzDdlMScHQgmXotC47wOPtpnhOXfEtmGx4PRVlNI3EUzSxgmGGFjX0QIBzR7n3s609wDhm6tytKMRb859Jwy5nlnIkSBE-Cb77JtStt1oYa3rHwjcmK1sQRAgUzSrhOMv9JF6VDLUhBUXrX2QmQ","text":"AI 介紹明新科大,20字以內"},"webhookEventId":"01KS6SKD97TR6C91V8YSVTC4TR","deliveryContext":{"isRedelivery":false},"timestamp":1779418379071,"source":{"type":"user","userId":"Uc736f30f5abec3489b5e495c080336e4"},"replyToken":"5c41c784896d45a488a9a3ebca24f913","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 02:53:05] "POST / HTTP/1.1" 200 -


BODY:  {"destination":"U3169a962e22ea2afc45f6eb98941b9c1","events":[{"type":"message","message":{"type":"text","id":"615031076557685271","quoteToken":"aeWethbA1xdbgYEaT8c6ElOqnx7dXHXrixeT250rIHrhR9REvJQt72wMBnU_OgcxT87a3TcNfU5kDxh4WfZuXooFBCtSbzvLUAbfuXUtb3XWIFtdN_9nhRmT8TL1YooqIcz6vsSNJG4xVU32DsftxA","markAsReadToken":"qd4_UuaX9ZWkvlUsLL3KG2VUSI9OVresQBWToiW3uI8mYPdp3ZZ1Dr54GUqMyM2HvoIry4PfEk39fR9l5UOtNfWu7WQ-tu5ulLxnEqIu6cQ-zq0DWuo7jpgfitTXL37ZdDCr-DujF8j1nM7RsCDHApGv-1r5szsZNLsu7DAsN3sQnwPlMQ1Zm03RaMAtK_6eUdxYMln9u0UTPDfdwS973Q","text":"AI 校長是誰"},"webhookEventId":"01KS6SKT9E0CMJA6FHC7R8N7G1","deliveryContext":{"isRedelivery":false},"timestamp":1779418392639,"source":{"type":"user","userId":"Uc736f30f5abec3489b5e495c080336e4"},"replyToken":"4913a30b8fae4354b7856da48e28c139","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 02:53:14] "POST / HTTP/1.1" 200 -
